# Week 4 Day 4 - Deep Agents (with playwright mcp)


In [ ]:
# Imports and environment first

import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_community.tools import GoogleSerperRun
from langchain_community.utilities import GoogleSerperAPIWrapper
from deepagents import create_deep_agent
from deepagents.backends import FilesystemBackend
from langchain_mcp_adapters.client import MultiServerMCPClient

load_dotenv(override=True)

In [ ]:
search = GoogleSerperRun(api_wrapper=GoogleSerperAPIWrapper())

In [ ]:
sandbox = os.path.abspath("sandbox")
os.makedirs(sandbox, exist_ok=True)

model = ChatOpenAI(model="gpt-5.4-mini")

researcher = create_deep_agent(
    model=model,
    tools=[search],
    system_prompt=(
        "You are a research analyst. Plan your work with your todo tool, "
        "research with the search tool, and write your findings as a tidy markdown briefing to a file."
    ),
    backend=FilesystemBackend(
        root_dir=sandbox, virtual_mode=True
    ),  # we are letting the agent know that it has access to this sandbox folder and can write files to
)

In [ ]:
researcher  # shows the langgraph graph when this cell is executed.

In [ ]:
brief = """
Our company is planning to move its sales fleet to electric vehicles.
Research the public EV charging landscape in Singapore: find out roughly how many public charging points there are,
and pick out two major charging networks a fleet could rely on.
Write a one page markdown briefing, with a heading and a short section for each, to the file charging.md.
"""
# brief is the end goal for the agent.
result = researcher.invoke({"messages": [{"role": "user", "content": brief}]})
print(result["messages"][-1].content)

In [ ]:
tools_used = [
    tc["name"]
    for m in result["messages"]
    for tc in (getattr(m, "tool_calls", []) or [])
]
print("Tools the agent called, in order:")
print(tools_used)

## MCP tools!


In [ ]:
# MCP TOOLS!

client = MultiServerMCPClient(
    {
        "playwright": {
            "transport": "stdio",
            "command": "npx",
            "args": ["-y", "@playwright/mcp@latest", "--isolated"],
        }
    }
)

browser_tools = await client.get_tools()
print(f"Loaded {len(browser_tools)} browser tools:")
for t in browser_tools:
    print(" -", t.name)

# providing the agent with too many tools can lead to problems - the agent can generate the WRONG INPUTS AND FAIL THE TOOL CALL, OR INPUTS THAT DON'T REALLY MATCH THE ONES REQUIRED, EG. GENERATING MORE THAN 1 REQUIRED INPUT.

research_browser_tools = [
    tool
    for tool in browser_tools
    if tool.name
    in {
        "browser_navigate",
        "browser_snapshot",
        # page-reading tool(s)
    }
]

In [ ]:
research_ev_instructions = """
You research one electric vehicle using the search tool and return three concise facts
that a fleet buyer would care about, such as price, range and charging.
"""

overall_instructions = """
You write comparison briefings for a company choosing electric vehicles for its sales fleet.
For each vehicle, delegate the research to your vehicle-researcher sub-agent,
then write a markdown comparison to a file, ending with a clear recommendation.
IMPORTANT: Use Playwright browser tools ONLY for HTTP/HTTPS web pages returned by the search tool.
DO NOT use Playwright tools to read, write, upload, or access local files.
Use the agent filesystem tools for local files such as fleet.md.
"""

# this is literally how we define the sub agent
research_subagent = {
    "name": "vehicle-researcher",
    "description": "Researches a single electric vehicle and returns a short list of facts about it.",
    "system_prompt": research_ev_instructions,
}

lead = create_deep_agent(
    model=model,
    tools=[
        search,
        *research_browser_tools,
    ],  # the tools must be a list, so we unpack the browser tools list
    system_prompt=overall_instructions,
    subagents=[research_subagent],  # define the sub agent here!
    backend=FilesystemBackend(root_dir=sandbox, virtual_mode=True),
)

In [ ]:
lead

In [ ]:
mission = """
Compare the Tesla Model Y and the Ford Mustang Mach-E as candidates for our 100-car sales fleet.
You can use the playwright browser tools to get the web contents of the links surfaced from the search tool for more information. Include the links that you use the browser tools on in the brief.
Research each vehicle, then write a short markdown comparison with a recommendation to fleet.md.
"""

result = await lead.ainvoke(
    {"messages": [{"role": "user", "content": mission}]}
)  # MUST use ainvoke here, in order to use the browser tools

In [ ]:
tools_used = [
    tc["name"]
    for m in result["messages"]
    for tc in (getattr(m, "tool_calls", []) or [])
]
print("Tools the lead agent called:", tools_used)

# task - means it has delegated to the sub-agent

### Now go and look at `fleet.md` in the Sandbox folder


In [ ]:
from langchain_core.tools import tool
from slide_kit import build_slide


@tool
def create_slide(title: str, key_points: list[str], recommendation: str) -> str:
    """Create a one-slide PowerPoint in the Voltway Research house style, saved as fleet.pptx."""
    build_slide(title, key_points, recommendation, os.path.join(sandbox, "fleet.pptx"))
    return "Saved the slide to /fleet.pptx"


# creating a sub-agent called slide_maker
slide_maker = {
    "name": "slide-maker",
    "description": "Turns a finished recommendation into a one-slide PowerPoint deck.",
    "system_prompt": "You turn research recommendations into slides, following your fleet-slide skill.",
    "tools": [create_slide],
    "skills": [
        "/skills/"
    ],  # we specify a skills directory which it has access to in its file system. It will read the skills.md file in the sub-folder in its skills directory
}

presenter = create_deep_agent(
    model=model,
    subagents=[slide_maker],
    system_prompt="You prepare research for presentation by delegating to your slide-maker sub-agent.",
    backend=FilesystemBackend(root_dir=sandbox, virtual_mode=True),
)

In [ ]:
result = presenter.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "Read fleet.md and have a one-slide deck made of its recommendation.",
            }
        ]
    }
)
print(result["messages"][-1].content)